In [ ]:
# Install dependencies if needed
# !pip install langchain langchain-experimental langchain-chroma pillow open_clip_torch torch matplotlib unstructured pydantic
import os
from textbook_loading import (
    load_book,
    clean_and_categorize_elements,
    summarize_elements,
    store_in_chromadb,
    delete_irrelevant_images,
)

In [ ]:
pdf_file = './data/first_n_second_Chapter_EmergencyInfectiveDisease.pdf'
image_output_dir = './figures/EmergencyInfectiveDisease'
chroma_persist_dir = './chroma/EmergencyInfectiveDisease/'

# Make sure the data directory exists
assert os.path.exists('./data'), "Error: './data' directory not found."
assert os.path.exists(pdf_file), f"Error: PDF file not found at {pdf_file}."

In [ ]:
print("📝 Unstructuring textbooks, filtering junks, semanic chunking...")
raw_pdf_elements = load_book(pdf_file, image_output_dir)
print("🎉 1.process_pdf_with_semantic_chunking complete.")


In [ ]:
# Clean and categorize
texts, tables, images_raw = clean_and_categorize_elements(raw_pdf_elements, window_size=2, min_meaningful_text_length=75)

In [ ]:
# Summarize, store, etc.
text_summaries, table_summaries, image_paths, relevant_images_to_summarize, image_summaries = summarize_elements(
    texts, tables, images_raw
)

In [ ]:
retriever = store_in_chromadb(
    text_summaries, texts, table_summaries, tables, image_paths,
    relevant_images_to_summarize, image_summaries,
    persist_directory=chroma_persist_dir
)

In [ ]:
delete_irrelevant_images(images_raw, relevant_images_to_summarize)

In [ ]:
# System sound, when done
sound_file = "/System/Library/Sounds/Glass.aiff"
os.system(f"afplay '{sound_file}'")

# Inspecting Retrieved Docs

In [ ]:
query = "show me a handsome cat?"
results = retriever.retrieve_multi_modal(query, k=5)

In [ ]:
from IPython.display import display, HTML
import os

# 1. Display all images together as thumbnails
image_paths = set()
for res in results:
    if res["modality"] == "image" and os.path.exists(res["summary"]):
        image_paths.add(res["summary"])
    elif res["modality"] == "image_summary":
        img_path = res["original_metadata"].get("image_path")
        if img_path and os.path.exists(img_path):
            image_paths.add(img_path)

if image_paths:
    html_imgs = " ".join(
        f'<img src="{img}" width="100" style="margin:2px; border:1px solid #ccc;">' for img in image_paths
    )
    display(HTML(html_imgs))
else:
    print("No images found in results.")

# 2. Display original text for each text result
print('-'*40, "Retrieved Text Chunks (first 300 chars)", '-'*40)
for res in results:
    if res["modality"] == "text":
        doc_id = res["original_metadata"].get("doc_id")
        original_text = None
        if doc_id and hasattr(retriever, "docstore"):
            doc = retriever.docstore._collection.get(ids=[doc_id], include=["documents"])
            if doc and doc.get("documents") and doc["documents"][0]:
                original_text = doc["documents"][0]
        if not original_text:
            original_text = res["summary"]
        text_display = original_text[:300] + ("..." if len(original_text) > 300 else "")
        print(text_display)
        print('-'*20)